# World Bank Data Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook retrieves development indicators from the World Bank API — nearly 16,000 time series covering population, GDP, poverty, health, education, gender, environment, and governance across 200+ countries. Select indicators by research preset or code, specify countries and date ranges, and get structured data for export.

The World Bank Indicators API is free, requires no API key or authentication, and provides data spanning 50+ years for most countries. This notebook makes that data accessible for research without requiring knowledge of indicator codes or API query construction.

## Key Features

1. **No API Key Required**: Completely open access — no registration or authentication
2. **Research Presets**: Pre-built indicator groups for demographics, economy, poverty, health, education, gender, environment, governance
3. **16,000+ Indicators**: Access any World Bank indicator by code
4. **Multi-Country**: Compare data across countries, regions, or income groups
5. **Date Ranges**: Retrieve data spanning 1960 to present
6. **Visualization**: Time series charts with multi-country comparison
7. **Structured Export**: CSV, Excel, and JSON with metadata

## Workflow

1. **Setup**: Install packages (no API key needed)
2. **Configure**: Select indicators, countries, and date range
3. **Retrieve**: Fetch data from the World Bank API
4. **Review**: Browse results with time series charts
5. **Export**: Download structured data

## Applications

This notebook supports research that requires international development context — comparing socioeconomic conditions across fieldwork countries, tracking development indicators over time, contextualizing ethnographic findings within global trends, or selecting field sites based on demographic and economic characteristics.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using international development statistics to complement ethnographic research. World Bank data reflects the categories and measurement frameworks of international development institutions. Researchers should consider what these indicators measure, what they omit, and how their construction shapes understanding of development.

**Important**: Development indicators are constructed measures with political and epistemological implications. GDP, poverty lines, and governance indices embed particular assumptions about progress and well-being.

## Target Audience

Designed for anthropologists and qualitative researchers who need international comparative data — from graduate students contextualizing fieldwork sites to applied researchers working across multiple countries.

## Technical Approach

The notebook queries the World Bank Indicators API v2 directly via HTTP requests. No authentication required. Responses in JSON with automatic pagination. Data returned as pandas DataFrames for analysis and export.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of computational notebooks for AI-assisted qualitative research. Alongside the toolkit's other structural-data explorers, it helps situate ethnography in structural context — the demographic, economic, and development conditions that ground a field site in its larger circumstances.

## License & Attribution

This notebook is released under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Anthropological Forum. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install supporting libraries. No API key or registration required — the World Bank API is completely open.

In [ ]:
# Install required packages
!pip install -q requests pandas matplotlib openpyxl ipywidgets

import re
import json
import unicodedata
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print("\u2713 All packages loaded")

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    pass

BASE_URL = 'https://api.worldbank.org/v2'

# --- Indicator Presets ---

INDICATOR_PRESETS = {
    'Demographics': {
        'description': 'Population, urbanization, life expectancy, fertility',
        'indicators': {
            'SP.POP.TOTL': 'Population, total',
            'SP.POP.GROW': 'Population growth (annual %)',
            'SP.URB.TOTL.IN.ZS': 'Urban population (% of total)',
            'SP.DYN.LE00.IN': 'Life expectancy at birth (years)',
            'SP.DYN.TFRT.IN': 'Fertility rate (births per woman)',
            'SP.DYN.CDRT.IN': 'Death rate (per 1,000)',
            'SM.POP.NETM': 'Net migration',
        },
    },
    'Economy': {
        'description': 'GDP, GNI, growth, inflation, trade',
        'indicators': {
            'NY.GDP.MKTP.CD': 'GDP (current US$)',
            'NY.GDP.PCAP.CD': 'GDP per capita (current US$)',
            'NY.GDP.MKTP.KD.ZG': 'GDP growth (annual %)',
            'NY.GNP.PCAP.CD': 'GNI per capita (current US$)',
            'FP.CPI.TOTL.ZG': 'Inflation, consumer prices (annual %)',
            'NE.TRD.GNFS.ZS': 'Trade (% of GDP)',
            'BX.KLT.DINV.CD.WD': 'Foreign direct investment, net inflows (US$)',
        },
    },
    'Poverty & Inequality': {
        'description': 'Poverty rates, Gini coefficient, income shares',
        'indicators': {
            'SI.POV.DDAY': 'Poverty headcount ratio at $2.15/day (%)',
            'SI.POV.LMIC': 'Poverty headcount ratio at $3.65/day (%)',
            'SI.POV.UMIC': 'Poverty headcount ratio at $6.85/day (%)',
            'SI.POV.GINI': 'Gini index',
            'SI.DST.10TH.10': 'Income share held by top 10%',
            'SI.DST.FRST.10': 'Income share held by lowest 10%',
        },
    },
    'Health': {
        'description': 'Mortality, disease, health infrastructure',
        'indicators': {
            'SH.DYN.MORT': 'Under-5 mortality rate (per 1,000)',
            'SH.STA.MMRT': 'Maternal mortality ratio (per 100,000)',
            'SH.IMM.MEAS': 'Immunization, measles (% of children)',
            'SH.MED.PHYS.ZS': 'Physicians (per 1,000 people)',
            'SH.XPD.CHEX.GD.ZS': 'Current health expenditure (% of GDP)',
            'SH.HIV.INCD.ZS': 'Incidence of HIV (per 1,000)',
            'SH.STA.BASS.ZS': 'People using basic sanitation services (%)',
        },
    },
    'Education': {
        'description': 'Enrollment, literacy, expenditure',
        'indicators': {
            'SE.ADT.LITR.ZS': 'Literacy rate, adult total (%)',
            'SE.PRM.ENRR': 'School enrollment, primary (% gross)',
            'SE.SEC.ENRR': 'School enrollment, secondary (% gross)',
            'SE.TER.ENRR': 'School enrollment, tertiary (% gross)',
            'SE.XPD.TOTL.GD.ZS': 'Government expenditure on education (% of GDP)',
            'SE.PRM.CMPT.ZS': 'Primary completion rate (%)',
        },
    },
    'Gender': {
        'description': 'Gender parity, labor participation, maternal health',
        'indicators': {
            'SL.TLF.CACT.FE.ZS': 'Labor force participation rate, female (%)',
            'SL.TLF.CACT.MA.ZS': 'Labor force participation rate, male (%)',
            'SE.ENR.PRIM.FM.ZS': 'School enrollment, primary (gender parity index)',
            'SG.GEN.PARL.ZS': 'Proportion of seats held by women in parliament (%)',
            'SP.ADO.TFRT': 'Adolescent fertility rate (per 1,000 women ages 15-19)',
        },
    },
    'Environment': {
        'description': 'Emissions, forest, water, energy',
        'indicators': {
            'EN.ATM.CO2E.PC': 'CO2 emissions (metric tons per capita)',
            'AG.LND.FRST.ZS': 'Forest area (% of land area)',
            'EG.USE.ELEC.KH.PC': 'Electric power consumption (kWh per capita)',
            'EG.FEC.RNEW.ZS': 'Renewable energy consumption (%)',
            'ER.H2O.FWTL.ZS': 'Annual freshwater withdrawals (% of internal)',
            'AG.LND.ARBL.ZS': 'Arable land (% of land area)',
        },
    },
    'Governance': {
        'description': 'Government effectiveness, rule of law, corruption',
        'indicators': {
            'GE.EST': 'Government Effectiveness: Estimate',
            'RQ.EST': 'Regulatory Quality: Estimate',
            'RL.EST': 'Rule of Law: Estimate',
            'CC.EST': 'Control of Corruption: Estimate',
            'PV.EST': 'Political Stability: Estimate',
            'VA.EST': 'Voice and Accountability: Estimate',
        },
    },
}

# Common country groups
COUNTRY_GROUPS = {
    'World': 'WLD',
    'Low income': 'LIC',
    'Lower middle income': 'LMC',
    'Upper middle income': 'UMC',
    'High income': 'HIC',
    'Sub-Saharan Africa': 'SSF',
    'Latin America & Caribbean': 'LCN',
    'South Asia': 'SAS',
    'East Asia & Pacific': 'EAS',
    'Middle East & North Africa': 'MEA',
    'Europe & Central Asia': 'ECS',
    'North America': 'NAC',
}

def make_slug(text):
    text = unicodedata.normalize('NFKD', text)
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[-\s]+', '_', text)[:50] or 'project'

def normalize_text(text):
    if not isinstance(text, str): return text
    for old, new in [('\u2011','-'),('\u2013','-'),('\u2014','-'),('\u2018',"'"),('\u2019',"'"),('\u201c','"'),('\u201d','"'),('\u2026','...')]:
        text = text.replace(old, new)
    return text

def style_excel(filepath):
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(left=Side(style='thin',color='E7ECEF'), right=Side(style='thin',color='E7ECEF'),
               top=Side(style='thin',color='E7ECEF'), bottom=Side(style='thin',color='E7ECEF'))
    for ws in wb.worksheets:
        for cell in ws[1]: cell.fill, cell.font, cell.alignment, cell.border = hf, hfont, Alignment(horizontal='center', vertical='center', wrap_text=True), tb
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max(max(len(str(c.value or '')) for c in col)+2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row: cell.alignment, cell.border = Alignment(vertical='top', wrap_text=True), tb
    wb.save(filepath)

print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"\u2713 {len(INDICATOR_PRESETS)} presets, {len(COUNTRY_GROUPS)} country groups")
print("\U0001f30d Ready to configure World Bank data retrieval!")

## Configuration

Select indicators, countries, and a date range. Use ISO 3166-1 alpha-2 country codes (e.g., US, BR, IN, NG) or select a regional/income group.

In [ ]:
# Configuration

class WBConfig:
    PRESET = 'Demographics'
    CUSTOM_INDICATORS = ''
    COUNTRIES = 'US, BR, IN, NG, DE'
    COUNTRY_GROUP = ''
    START_YEAR = 2000
    END_YEAR = 2024
    PROJECT_NAME = ''

config = WBConfig()

style = {'description_width': '160px'}
layout = widgets.Layout(width='500px')

def create_config_interface():

    config_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
    config_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f30d World Bank Data Explorer</h2>'
    config_html += '<p><strong>Welcome!</strong> This notebook retrieves development indicators from the World Bank API. No API key required.</p>'
    config_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
    config_html += '<ol>'
    config_html += '<li><strong>Configure:</strong> Select indicators, countries, and date range below</li>'
    config_html += '<li><strong>Retrieve:</strong> Fetch data from the World Bank API</li>'
    config_html += '<li><strong>Review:</strong> Browse data with charts</li>'
    config_html += '<li><strong>Export:</strong> Download structured results</li>'
    config_html += '</ol>'
    config_html += '</div>'

    instructions = widgets.HTML(value=config_html)

    ind_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Indicators</h3>')

    preset_input = widgets.Dropdown(
        options=[(f'{k} \u2014 {v["description"]}', k) for k, v in INDICATOR_PRESETS.items()] + [('Custom Indicator Codes', 'custom')],
        value='Demographics',
        description='Preset:', layout=layout, style=style
    )

    custom_indicators = widgets.Textarea(
        value='', placeholder='Enter World Bank indicator codes, one per line (e.g., SP.POP.TOTL)',
        description='Indicators:',
        layout=widgets.Layout(width='500px', height='80px'), style=style
    )
    custom_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Find indicator codes at <a href="https://data.worldbank.org/indicator" target="_blank">data.worldbank.org/indicator</a>.</p>'
    )
    custom_box = widgets.VBox([custom_indicators, custom_help])
    custom_box.layout.display = 'none'

    def on_preset_change(change):
        custom_box.layout.display = '' if change['new'] == 'custom' else 'none'
    preset_input.observe(on_preset_change, names='value')

    geo_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f30e Countries</h3>')

    countries_input = widgets.Text(
        value='US, BR, IN, NG, DE',
        placeholder='ISO2 codes, comma-separated (e.g., US, BR, IN)',
        description='Countries:', layout=layout, style=style
    )

    country_group_input = widgets.Dropdown(
        options=[('Specific countries (above)', '')] + [(k, v) for k, v in COUNTRY_GROUPS.items()],
        value='',
        description='Or group:', layout=layout, style=style
    )

    geo_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Enter ISO 3166-1 alpha-2 codes (US, GB, BR, IN, CN, NG, etc.) or select a regional/income group. '
        'Group selection overrides individual country codes.</p>'
    )

    date_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c5 Date Range</h3>')

    start_year = widgets.IntSlider(value=2000, min=1960, max=2025, description='Start year:', style=style, layout=layout)
    end_year = widgets.IntSlider(value=2024, min=1960, max=2025, description='End year:', style=style, layout=layout)

    project_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4cb Project</h3>')
    project_name = widgets.Text(value='', placeholder='e.g., Field Site Comparison', description='Project name:', layout=layout, style=style)

    save_btn = widgets.Button(description='Save Configuration', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='20px 0'))
    status = widgets.Output()

    def save_config(btn):
        config.PRESET = preset_input.value
        config.CUSTOM_INDICATORS = custom_indicators.value.strip()
        config.COUNTRIES = countries_input.value.strip()
        config.COUNTRY_GROUP = country_group_input.value
        config.START_YEAR = start_year.value
        config.END_YEAR = end_year.value
        config.PROJECT_NAME = project_name.value.strip()

        with status:
            clear_output()
            if config.PRESET == 'custom':
                n = len([i.strip() for i in config.CUSTOM_INDICATORS.split('\n') if i.strip()])
                print(f"\u2713 Custom indicators: {n}")
            else:
                n = len(INDICATOR_PRESETS[config.PRESET]['indicators'])
                print(f"\u2713 Preset: {config.PRESET} ({n} indicators)")

            if config.COUNTRY_GROUP:
                group_name = [k for k, v in COUNTRY_GROUPS.items() if v == config.COUNTRY_GROUP][0]
                print(f"\u2713 Country group: {group_name}")
            else:
                print(f"\u2713 Countries: {config.COUNTRIES}")
            print(f"\u2713 Date range: {config.START_YEAR}\u2013{config.END_YEAR}")
            print()
            print("\u2713 Configuration saved! Proceed to Retrieve.")

    save_btn.on_click(save_config)

    display(widgets.VBox([
        instructions,
        ind_header, preset_input, custom_box,
        geo_header, countries_input, country_group_input, geo_help,
        date_header, start_year, end_year,
        project_header, project_name,
        save_btn, status,
    ]))

create_config_interface()

## Retrieve Data

Fetch indicator data from the World Bank API. Each indicator is retrieved separately with automatic pagination.

In [ ]:
# Retrieve Data

import time

wb_df = None
indicator_labels = {}

def fetch_indicator(indicator_code, countries, start_year, end_year):
    """Fetch a single indicator from the World Bank API with pagination."""
    all_data = []
    page = 1

    while True:
        url = f'{BASE_URL}/country/{countries}/indicator/{indicator_code}'
        params = {
            'date': f'{start_year}:{end_year}',
            'format': 'json',
            'per_page': 1000,
            'page': page,
        }

        response = None
        for attempt in range(3):
            try:
                response = requests.get(url, params=params, timeout=40)
                break
            except requests.exceptions.Timeout:
                if attempt == 2:
                    raise RuntimeError("World Bank API timed out after 3 attempts — retry later")
                time.sleep(3)
        if response.status_code != 200:
            raise RuntimeError(f"API returned status {response.status_code}")

        data = response.json()

        if not isinstance(data, list) or len(data) < 2:
            break

        metadata = data[0]
        records = data[1]

        if not records:
            break

        all_data.extend(records)

        total_pages = metadata.get('pages', 1)
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.3)

    return all_data


retrieve_btn = widgets.Button(
    description='Retrieve Data',
    style={'button_color': '#6096BA'},
    layout=widgets.Layout(width='200px', margin='10px 0')
)

progress_html = widgets.HTML(value='')
retrieve_out = widgets.Output()

def on_retrieve(_):
    global wb_df, indicator_labels
    retrieve_out.clear_output()
    wb_df = None

    # Determine indicators
    if config.PRESET == 'custom':
        codes = [i.strip() for i in config.CUSTOM_INDICATORS.split('\n') if i.strip()]
        labels = {c: c for c in codes}
    else:
        preset = INDICATOR_PRESETS[config.PRESET]
        labels = preset['indicators']
        codes = list(labels.keys())

    indicator_labels = labels

    # Determine countries
    if config.COUNTRY_GROUP:
        countries = config.COUNTRY_GROUP
    else:
        countries = ';'.join(c.strip() for c in config.COUNTRIES.split(',') if c.strip())

    with retrieve_out:
        print(f"\U0001f30d Fetching {len(codes)} indicator(s) from World Bank API...")
        print(f"   Date range: {config.START_YEAR}\u2013{config.END_YEAR}")
        print()

    all_rows = []
    for idx, code in enumerate(codes):
        label = labels.get(code, code)
        progress_html.value = f'<span style="color: #274C77;">Fetching {idx+1}/{len(codes)} \u2014 {label}</span>'

        try:
            records = fetch_indicator(code, countries, config.START_YEAR, config.END_YEAR)

            for rec in records:
                if rec.get('value') is not None:
                    all_rows.append({
                        'indicator_code': code,
                        'indicator_name': label,
                        'country_code': rec.get('countryiso3code', rec.get('country', {}).get('id', '')),
                        'country_name': rec.get('country', {}).get('value', ''),
                        'year': int(rec.get('date', 0)),
                        'value': float(rec['value']),
                    })

            with retrieve_out:
                n = sum(1 for r in records if r.get('value') is not None)
                print(f"\u2713 {label}: {n} observations")

        except Exception as e:
            with retrieve_out:
                print(f"\u274c {label}: {e}")

    progress_html.value = ''

    if not all_rows:
        with retrieve_out:
            print("\n\u26a0\ufe0f No data returned.")
        return

    wb_df = pd.DataFrame(all_rows)
    wb_df.sort_values(['indicator_name', 'country_name', 'year'], inplace=True)
    wb_df.reset_index(drop=True, inplace=True)

    with retrieve_out:
        print()
        print(f"\u2713 Retrieved {len(wb_df)} total observations")
        print(f"   Indicators: {wb_df['indicator_code'].nunique()}")
        print(f"   Countries: {wb_df['country_name'].nunique()}")
        print(f"   Years: {wb_df['year'].min()}\u2013{wb_df['year'].max()}")

retrieve_btn.on_click(on_retrieve)
display(widgets.VBox([retrieve_btn, progress_html, retrieve_out]))

## Review Data

Browse results with time series charts comparing countries across indicators.

In [ ]:
# Review Data

COLORS = ['#274C77', '#6096BA', '#A3CEF1', '#8B8C89', '#4A7C59', '#CC6644', '#7B5EA7', '#C9A227']

if wb_df is None:
    print("\u26a0\ufe0f Run the Retrieve cell first.")
else:
    indicators = wb_df['indicator_name'].unique()
    countries = wb_df['country_name'].unique()

    # One chart per indicator
    for ind_name in indicators:
        ind_data = wb_df[wb_df['indicator_name'] == ind_name]

        fig, ax = plt.subplots(figsize=(10, 5))
        fig.patch.set_facecolor('#FFFFFF')
        ax.set_facecolor('#FAFAFA')

        for i, country in enumerate(countries):
            subset = ind_data[ind_data['country_name'] == country].sort_values('year')
            if subset.empty:
                continue
            color = COLORS[i % len(COLORS)]
            ax.plot(subset['year'], subset['value'], label=country, color=color, linewidth=1.5, marker='o', markersize=3)

        ax.set_title(ind_name, fontsize=13, color='#274C77', fontweight='bold', pad=10)
        ax.set_xlabel('Year', fontsize=10, color='#274C77')
        ax.legend(frameon=True, facecolor='white', edgecolor='#E7ECEF', fontsize=8)
        ax.grid(True, alpha=0.3, color='#8B8C89')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#E7ECEF')
        ax.spines['bottom'].set_color('#E7ECEF')

        plt.tight_layout()
        plt.show()

    # Summary
    print(f"\U0001f4cb Summary: {len(indicators)} indicators, {len(countries)} countries, {len(wb_df)} observations")
    print()
    display(wb_df.head(20))

## Export

Export World Bank data as CSV, Excel, or JSON.

In [ ]:
# Export

if wb_df is None:
    print("\u26a0\ufe0f Run the Retrieve cell first.")
else:
    export_format = widgets.Dropdown(
        options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx) \u2014 styled with metadata', 'excel'), ('JSON (.json)', 'json')],
        value='excel', description='Format:', style={'description_width': '80px'}, layout=widgets.Layout(width='500px')
    )

    export_btn = widgets.Button(description='Export', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='10px 0'))
    export_out = widgets.Output()

    def on_export(_):
        export_out.clear_output()
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(config.PROJECT_NAME) if config.PROJECT_NAME else 'worldbank'
        preset_slug = make_slug(config.PRESET)[:15]
        fmt = export_format.value

        try:
            if fmt == 'csv':
                filepath = f'{slug}_{preset_slug}_{timestamp}.csv'
                wb_df.to_csv(filepath, index=False)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB: colab_files.download(filepath)

            elif fmt == 'excel':
                filepath = f'{slug}_{preset_slug}_{timestamp}.xlsx'
                with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                    wb_df.to_excel(writer, sheet_name='Data', index=False)
                    meta_rows = [
                        {'Field': 'Preset', 'Value': config.PRESET},
                        {'Field': 'Countries', 'Value': config.COUNTRIES if not config.COUNTRY_GROUP else config.COUNTRY_GROUP},
                        {'Field': 'Date Range', 'Value': f'{config.START_YEAR}-{config.END_YEAR}'},
                        {'Field': 'Indicators', 'Value': wb_df['indicator_code'].nunique()},
                        {'Field': 'Countries', 'Value': wb_df['country_name'].nunique()},
                        {'Field': 'Observations', 'Value': len(wb_df)},
                        {'Field': 'Retrieved', 'Value': datetime.now().isoformat()},
                    ]
                    if config.PROJECT_NAME:
                        meta_rows.insert(0, {'Field': 'Project', 'Value': config.PROJECT_NAME})
                    pd.DataFrame(meta_rows).to_excel(writer, sheet_name='Metadata', index=False)
                style_excel(filepath)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB: colab_files.download(filepath)

            elif fmt == 'json':
                filepath = f'{slug}_{preset_slug}_{timestamp}.json'
                output = {
                    'metadata': {
                        'project_name': config.PROJECT_NAME,
                        'preset': config.PRESET,
                        'start_year': config.START_YEAR,
                        'end_year': config.END_YEAR,
                        'n_indicators': wb_df['indicator_code'].nunique(),
                        'n_countries': wb_df['country_name'].nunique(),
                        'n_observations': len(wb_df),
                        'retrieved_at': datetime.now().isoformat(),
                    },
                    'indicator_labels': indicator_labels,
                    'data': json.loads(wb_df.to_json(orient='records')),
                }
                with open(filepath, 'w') as f:
                    json.dump(output, f, indent=2, default=str)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB: colab_files.download(filepath)

        except Exception as e:
            with export_out:
                print(f"\u274c Export error: {type(e).__name__}: {e}")

    export_btn.on_click(on_export)
    display(widgets.VBox([export_format, export_btn, export_out]))